In [1]:
import os
import math
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torchaudio.transforms as T
from torch.utils.data import DataLoader, Dataset
from pathlib import Path

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


In [2]:
DATA_ROOT = r'G:\.shortcut-targets-by-id\1ua-L70AEx_VqEEKdgxjSMzr-wEX6GnxK\voice-manipulation-detection'
CACHE_DIR = './lfcc_cache'
SUBSET_FRACTION = 1
MAX_LENGTH = 64000
BATCH_SIZE = 32

CHECKPOINTS = {
    'cnn': './models/checkpoints/cnn/cnn_best_model.pt',
    'rnn': './models/checkpoints/rnn/rnn_best_model.pt',
    'transformer': './models/checkpoints/transformer/transformer_best_model.pt'
}

for name, path in CHECKPOINTS.items():
    exists = os.path.exists(path)
    status = 'FOUND' if exists else 'MISSING'
    print(f"{name}: {status} -> {path}")

cnn: FOUND -> ./models/checkpoints/cnn/cnn_best_model.pt
rnn: FOUND -> ./models/checkpoints/rnn/rnn_best_model.pt
transformer: FOUND -> ./models/checkpoints/transformer/transformer_best_model.pt


In [3]:
class ASVspoofDataset(Dataset):
    def __init__(self, data_root, split='eval', cache_dir='./lfcc_cache', subset_fraction=1.0, max_length=64000):
        import soundfile as sf
        import librosa
        self.sf = sf
        self.librosa = librosa
        self.root_dir = data_root
        self.cache_dir = Path(cache_dir) / split
        self.cache_dir.mkdir(parents=True, exist_ok=True)

        protocol_dir = os.path.join(self.root_dir, 'data', 'raw', 'ASVspoof2019', 'LA', 'LA', 'ASVspoof2019_LA_cm_protocols')
        protocol_files = {'train': 'ASVspoof2019.LA.cm.train.trn.txt', 'dev': 'ASVspoof2019.LA.cm.dev.trl.txt', 'eval': 'ASVspoof2019.LA.cm.eval.trl.txt'}
        protocol_path = os.path.join(protocol_dir, protocol_files[split])

        if not os.path.exists(protocol_path):
            raise FileNotFoundError(f"Protocol file not found: {protocol_path}")

        self.metadata = pd.read_csv(protocol_path, sep=' ', header=None, names=['speaker', 'filename', 'system', 'null', 'label'])

        if subset_fraction < 1.0:
            self.metadata = self.metadata.sample(frac=subset_fraction, random_state=42).reset_index(drop=True)

        self.audio_dir = os.path.join(self.root_dir, 'data', 'raw', 'ASVspoof2019', 'LA', 'LA', f'ASVspoof2019_LA_{split}', 'flac')
        self.max_length = max_length

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, idx):
        row = self.metadata.iloc[idx]
        cache_path = self.cache_dir / f"{row['filename']}.pt"

        if cache_path.exists():
            try:
                data = torch.load(cache_path, weights_only=True)
            except TypeError:
                data = torch.load(cache_path)
            return {
                'lfcc': data['lfcc'],
                'label': torch.tensor(data['label'], dtype=torch.long) if not isinstance(data['label'], torch.Tensor) else data['label'],
                'is_lfcc': True,
                'filename': row['filename'],
                'system': row['system']
            }

        audio_path = os.path.join(self.audio_dir, row['filename'] + '.flac')
        try:
            audio_data, sample_rate = self.sf.read(audio_path)
            if len(audio_data.shape) > 1:
                audio_data = audio_data.mean(axis=1)
            if sample_rate != 16000:
                audio_data = self.librosa.resample(audio_data, orig_sr=sample_rate, target_sr=16000)
            waveform = torch.from_numpy(audio_data).float().unsqueeze(0)
            if waveform.shape[1] > self.max_length:
                waveform = waveform[:, :self.max_length]
            else:
                pad = self.max_length - waveform.shape[1]
                waveform = torch.nn.functional.pad(waveform, (0, pad))
        except:
            waveform = torch.zeros(1, self.max_length)

        label = 0 if row['label'] == 'bonafide' else 1
        return {
            'waveform': waveform,
            'label': torch.tensor(label, dtype=torch.long),
            'is_lfcc': False,
            'filename': row['filename'],
            'system': row['system']
        }

In [4]:
class LFCCExtractor(nn.Module):
    def __init__(self, sample_rate=16000, n_lfcc=60, n_fft=512, hop_length=160):
        super().__init__()
        self.n_lfcc = n_lfcc
        self.spec = T.Spectrogram(n_fft=n_fft, hop_length=hop_length, power=2.0)
        self.register_buffer('dct_mat', None)

    def _create_dct_matrix(self, n_freqs, n_lfcc):
        n = torch.arange(float(n_freqs)).unsqueeze(1)
        k = torch.arange(float(n_lfcc)).unsqueeze(0)
        dct = torch.cos(torch.pi / float(n_freqs) * (n + 0.5) * k)
        return dct / torch.sqrt(torch.sum(dct**2, dim=0, keepdim=True))

    def forward(self, waveform):
        spec = self.spec(waveform).squeeze(1)
        spec = torch.log(torch.sqrt(spec) + 1e-10)
        if self.dct_mat is None:
            self.dct_mat = self._create_dct_matrix(spec.shape[1], self.n_lfcc).to(spec.device)
        spec = spec.transpose(1, 2)
        return torch.matmul(spec, self.dct_mat).transpose(1, 2)


def make_collate(lfcc_extractor, device, model_type='cnn'):
    transpose = model_type in ('rnn', 'transformer')

    def collate_fn(batch):
        labels = torch.stack([b['label'] for b in batch]).to(device)
        filenames = [b['filename'] for b in batch]
        systems = [b['system'] for b in batch]

        if batch[0].get('is_lfcc', False):
            features = torch.stack([b['lfcc'] for b in batch]).to(device)
            if transpose:
                features = features.transpose(1, 2)
            return features, labels, filenames, systems

        waveforms = torch.stack([b['waveform'] for b in batch]).to(device)
        with torch.no_grad():
            features = lfcc_extractor(waveforms)
            if transpose:
                features = features.transpose(1, 2)
        return features, labels, filenames, systems

    return collate_fn


lfcc_extractor = LFCCExtractor(n_lfcc=60).to(device)
print("LFCC extractor ready")

LFCC extractor ready


In [5]:
# --- CNN ---
class ResidualBlock(nn.Module):
    def __init__(self, channels, kernel_size=3):
        super().__init__()
        self.conv1 = nn.Conv1d(channels, channels, kernel_size, padding=kernel_size//2)
        self.bn1 = nn.BatchNorm1d(channels)
        self.conv2 = nn.Conv1d(channels, channels, kernel_size, padding=kernel_size//2)
        self.bn2 = nn.BatchNorm1d(channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        residual = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out)) + residual
        return self.relu(out)

class ImprovedSpoofDetector(nn.Module):
    def __init__(self, n_lfcc=60, hidden_channels=128, num_classes=2, dropout=0.3):
        super().__init__()
        self.conv_input = nn.Sequential(
            nn.Conv1d(n_lfcc, hidden_channels, kernel_size=7, padding=3),
            nn.BatchNorm1d(hidden_channels),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout)
        )
        self.res_blocks = nn.Sequential(
            ResidualBlock(hidden_channels, kernel_size=3),
            nn.MaxPool1d(2), nn.Dropout(dropout),
            ResidualBlock(hidden_channels, kernel_size=3),
            nn.MaxPool1d(2), nn.Dropout(dropout),
            ResidualBlock(hidden_channels, kernel_size=3),
            nn.MaxPool1d(2), nn.Dropout(dropout),
        )
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_channels, hidden_channels // 2),
            nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.Linear(hidden_channels // 2, num_classes)
        )

    def forward(self, lfcc_features):
        x = self.conv_input(lfcc_features)
        x = self.res_blocks(x)
        x = self.global_pool(x).squeeze(-1)
        return self.classifier(x)

# --- RNN ---
class BiLSTMSpoofDetector(nn.Module):
    def __init__(self, input_size=60, hidden_size=128, num_layers=2, num_classes=2, dropout=0.5, bidirectional=True):
        super().__init__()
        self.layer_norm = nn.LayerNorm(input_size)
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size, num_layers=num_layers,
                           batch_first=True, dropout=dropout if num_layers > 1 else 0, bidirectional=bidirectional)
        lstm_output_size = hidden_size * 2 if bidirectional else hidden_size
        self.attention = nn.Sequential(
            nn.Linear(lstm_output_size, lstm_output_size // 2), nn.Tanh(),
            nn.Linear(lstm_output_size // 2, 1)
        )
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(lstm_output_size, lstm_output_size // 2),
            nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.Linear(lstm_output_size // 2, num_classes)
        )

    def forward(self, lfcc_features):
        x = self.layer_norm(lfcc_features)
        lstm_out, _ = self.lstm(x)
        attn_weights = torch.softmax(self.attention(lstm_out), dim=1)
        context = torch.sum(attn_weights * lstm_out, dim=1)
        return self.classifier(context)

# --- Transformer ---
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=420, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

class TransformerSpoofDetector(nn.Module):
    def __init__(self, input_dim=60, d_model=128, nhead=4, num_layers=2, dim_feedforward=256, dropout=0.3, num_classes=2):
        super().__init__()
        self.input_projection = nn.Sequential(
            nn.Linear(input_dim, d_model), nn.LayerNorm(d_model), nn.Dropout(dropout)
        )
        self.pos_encoder = PositionalEncoding(d_model, max_len=420, dropout=dropout)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, activation='gelu', batch_first=True, norm_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers, norm=nn.LayerNorm(d_model))
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model // 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model // 2, num_classes)
        )

    def forward(self, lfcc_features):
        x = self.input_projection(lfcc_features)
        batch_size = x.size(0)
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)
        x = self.pos_encoder(x)
        x = self.transformer_encoder(x)
        cls_output = x[:, 0, :]
        return self.classifier(cls_output)

print("All model architectures loaded")

All model architectures loaded


In [6]:
def load_model(model_name, checkpoint_path, device):
    if model_name == 'cnn':
        model = ImprovedSpoofDetector(n_lfcc=60, hidden_channels=64, num_classes=2, dropout=0.5)
    elif model_name == 'rnn':
        model = BiLSTMSpoofDetector(input_size=60, hidden_size=128, num_layers=2, num_classes=2, dropout=0.5, bidirectional=True)
    elif model_name == 'transformer':
        model = TransformerSpoofDetector(input_dim=60, d_model=128, nhead=4, num_layers=2, dim_feedforward=256, dropout=0.3, num_classes=2)
    else:
        raise ValueError(f"Unknown model: {model_name}")

    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model = model.to(device)
    model.eval()
    print(f"Loaded {model_name.upper()} from epoch {checkpoint.get('epoch', '?')} | Val EER: {checkpoint.get('eer', '?')} | Val Acc: {checkpoint.get('accuracy', '?')}")
    return model, checkpoint


def calculate_eer(labels, scores):
    labels = labels.astype(int)
    scores = scores.astype(float)
    n_spoof = np.sum(labels == 1)
    n_bonafide = np.sum(labels == 0)
    if n_spoof == 0 or n_bonafide == 0:
        return 0.5
    order = np.argsort(-scores)
    labels_sorted = labels[order]
    tps = np.cumsum(labels_sorted == 1)
    fps = np.cumsum(labels_sorted == 0)
    tpr = tps / n_spoof
    fpr = fps / n_bonafide
    fnr = 1 - tpr
    idx = np.nanargmin(np.abs(fnr - fpr))
    eer = (fnr[idx] + fpr[idx]) / 2
    return max(0.0, min(1.0, float(eer)))


def evaluate_model(model, data_loader, device):
    model.eval()
    all_scores, all_labels, all_preds, all_systems = [], [], [], []
    correct, total = 0, 0

    with torch.no_grad():
        for data, target, filenames, systems in data_loader:
            output = model(data)
            probs = torch.softmax(output, dim=1)
            scores = probs[:, 1].cpu().numpy()
            _, predicted = torch.max(output, 1)
            total += target.size(0)
            correct += (predicted == target).sum().item()
            all_scores.extend(scores)
            all_labels.extend(target.cpu().numpy())
            all_preds.extend(predicted.cpu().numpy())
            all_systems.extend(systems)

    all_labels = np.array(all_labels)
    all_scores = np.array(all_scores)
    all_preds = np.array(all_preds)
    accuracy = 100 * correct / total
    eer = calculate_eer(all_labels, all_scores)

    tp = np.sum((all_preds == 1) & (all_labels == 1))
    tn = np.sum((all_preds == 0) & (all_labels == 0))
    fp = np.sum((all_preds == 1) & (all_labels == 0))
    fn = np.sum((all_preds == 0) & (all_labels == 1))
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    return {
        'accuracy': accuracy, 'eer': eer,
        'precision': precision, 'recall': recall, 'f1': f1,
        'tp': tp, 'tn': tn, 'fp': fp, 'fn': fn,
        'labels': all_labels, 'scores': all_scores, 'preds': all_preds,
        'systems': np.array(all_systems), 'total': total
    }

print("Evaluation functions ready")

Evaluation functions ready


In [7]:
print("Loading eval dataset...")
eval_dataset = ASVspoofDataset(DATA_ROOT, split='eval', cache_dir=CACHE_DIR, subset_fraction=SUBSET_FRACTION, max_length=MAX_LENGTH)

eval_labels = eval_dataset.metadata['label'].values
eval_bonafide = (eval_labels == 'bonafide').sum()
eval_spoof = (eval_labels == 'spoof').sum()
print(f"Eval set: {len(eval_dataset)} samples (Bonafide: {eval_bonafide}, Spoof: {eval_spoof})")

print("\nCaching eval LFCC features...")
cached = 0
for idx in range(len(eval_dataset)):
    row = eval_dataset.metadata.iloc[idx]
    cache_path = eval_dataset.cache_dir / f"{row['filename']}.pt"
    if cache_path.exists():
        cached += 1
        continue
    audio_path = os.path.join(eval_dataset.audio_dir, row['filename'] + '.flac')
    try:
        audio_data, sr = eval_dataset.sf.read(audio_path)
        if len(audio_data.shape) > 1:
            audio_data = audio_data.mean(axis=1)
        if sr != 16000:
            audio_data = eval_dataset.librosa.resample(audio_data, orig_sr=sr, target_sr=16000)
        waveform = torch.from_numpy(audio_data).float().unsqueeze(0)
        if waveform.shape[1] > MAX_LENGTH:
            waveform = waveform[:, :MAX_LENGTH]
        else:
            pad = MAX_LENGTH - waveform.shape[1]
            waveform = torch.nn.functional.pad(waveform, (0, pad))
    except:
        waveform = torch.zeros(1, MAX_LENGTH)

    waveform = waveform.to(device).unsqueeze(0)
    with torch.no_grad():
        lfcc = lfcc_extractor(waveform).squeeze(0).cpu()
    label = 0 if row['label'] == 'bonafide' else 1
    torch.save({'lfcc': lfcc, 'label': label}, cache_path)
    if (idx + 1) % 500 == 0:
        print(f"  Cached {idx + 1}/{len(eval_dataset)}", flush=True)

print(f"Eval cache: {cached} existing + {len(eval_dataset) - cached} new = {len(eval_dataset)} total")

Loading eval dataset...
Eval set: 71237 samples (Bonafide: 7355, Spoof: 63882)

Caching eval LFCC features...
Eval cache: 71237 existing + 0 new = 71237 total


In [8]:
print("="*60)
print("EVALUATING ALL MODELS ON HELD-OUT TEST SET")
print("="*60)

results = {}

for model_name in ['cnn', 'rnn', 'transformer']:
    checkpoint_path = CHECKPOINTS[model_name]
    if not os.path.exists(checkpoint_path):
        print(f"\n[SKIP] {model_name.upper()} - checkpoint not found")
        continue

    print(f"\n{'='*60}")
    print(f"Evaluating: {model_name.upper()}")
    print(f"{'='*60}")

    model, checkpoint = load_model(model_name, checkpoint_path, device)
    collate_fn = make_collate(lfcc_extractor, device, model_type=model_name)
    eval_loader = DataLoader(eval_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False, collate_fn=collate_fn)

    metrics = evaluate_model(model, eval_loader, device)
    results[model_name] = metrics

    print(f"\n  Accuracy:  {metrics['accuracy']:.2f}%")
    print(f"  EER:       {metrics['eer']:.4f} ({metrics['eer']*100:.2f}%)")
    print(f"  Precision: {metrics['precision']:.4f}")
    print(f"  Recall:    {metrics['recall']:.4f}")
    print(f"  F1 Score:  {metrics['f1']:.4f}")
    print(f"  Confusion: TP={metrics['tp']} TN={metrics['tn']} FP={metrics['fp']} FN={metrics['fn']}")
    print(f"  Samples:   {metrics['total']}")

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"\n{'='*60}")
print("All evaluations complete")
print(f"{'='*60}")

EVALUATING ALL MODELS ON HELD-OUT TEST SET

Evaluating: CNN
Loaded CNN from epoch 14 | Val EER: 0.006383630751207655 | Val Acc: 89.56285687540952


C:\Users\PC\AppData\Local\Temp\ipykernel_22112\493293170.py:11: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path, map_location=device)


KeyboardInterrupt: 

In [ ]:
print("\n" + "="*60)
print("SUMMARY TABLE")
print("="*60)
print(f"{'Model':<15} {'Accuracy':>10} {'EER':>10} {'Precision':>10} {'Recall':>10} {'F1':>10}")
print("-"*65)
for name, m in results.items():
    print(f"{name.upper():<15} {m['accuracy']:>9.2f}% {m['eer']:>9.4f} {m['precision']:>10.4f} {m['recall']:>10.4f} {m['f1']:>10.4f}")

best_model = min(results, key=lambda k: results[k]['eer'])
print(f"\nBest model by EER: {best_model.upper()} (EER={results[best_model]['eer']:.4f})")

In [ ]:
print("Per-Attack-Type Analysis")
print("="*60)

for model_name, metrics in results.items():
    print(f"\n--- {model_name.upper()} ---")
    systems = metrics['systems']
    labels = metrics['labels']
    preds = metrics['preds']
    scores = metrics['scores']

    unique_systems = sorted(set(systems))
    print(f"{'System':<10} {'Count':>8} {'Acc':>8} {'EER':>8}")
    print("-"*36)

    for sys in unique_systems:
        mask = systems == sys
        if mask.sum() == 0:
            continue
        sys_labels = labels[mask]
        sys_preds = preds[mask]
        sys_scores = scores[mask]
        sys_acc = 100 * np.mean(sys_labels == sys_preds)
        sys_eer = calculate_eer(sys_labels, sys_scores) if len(set(sys_labels)) > 1 else float('nan')
        print(f"{sys:<10} {mask.sum():>8} {sys_acc:>7.2f}% {sys_eer:>7.4f}")

In [ ]:
fig, axes = plt.subplots(1, len(results), figsize=(5*len(results), 4))
if len(results) == 1:
    axes = [axes]

for ax, (name, m) in zip(axes, results.items()):
    cm = np.array([[m['tn'], m['fp']], [m['fn'], m['tp']]])
    im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
    ax.set_title(f"{name.upper()}\nAcc={m['accuracy']:.1f}% EER={m['eer']:.4f}")
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(['Bonafide', 'Spoof'])
    ax.set_yticklabels(['Bonafide', 'Spoof'])
    for i in range(2):
        for j in range(2):
            color = 'white' if cm[i, j] > cm.max() / 2 else 'black'
            ax.text(j, i, str(cm[i, j]), ha='center', va='center', color=color, fontsize=14)

plt.tight_layout()
fig_dir = Path('./experiments/evaluation/figures')
fig_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(fig_dir / 'confusion_matrices.png', dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(results), figsize=(5*len(results), 4))
if len(results) == 1:
    axes = [axes]

for ax, (name, m) in zip(axes, results.items()):
    bonafide_scores = m['scores'][m['labels'] == 0]
    spoof_scores = m['scores'][m['labels'] == 1]
    ax.hist(bonafide_scores, bins=50, alpha=0.6, label='Bonafide', color='green', density=True)
    ax.hist(spoof_scores, bins=50, alpha=0.6, label='Spoof', color='red', density=True)
    ax.set_title(f"{name.upper()} Score Distribution")
    ax.set_xlabel('Spoof Score')
    ax.set_ylabel('Density')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(fig_dir / 'score_distributions.png', dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(7, 6))
colors = {'cnn': 'blue', 'rnn': 'orange', 'transformer': 'green'}

for name, m in results.items():
    labels = m['labels'].astype(int)
    scores = m['scores'].astype(float)
    n_spoof = np.sum(labels == 1)
    n_bonafide = np.sum(labels == 0)

    thresholds = np.sort(scores)
    fpr_list, fnr_list = [], []
    for t in thresholds[::max(1, len(thresholds)//200)]:
        fp = np.sum((scores >= t) & (labels == 0))
        fn = np.sum((scores < t) & (labels == 1))
        fpr_list.append(fp / n_bonafide)
        fnr_list.append(fn / n_spoof)

    ax.plot(fpr_list, fnr_list, label=f"{name.upper()} (EER={m['eer']:.4f})", color=colors.get(name, 'black'), linewidth=2)

ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='EER line')
ax.set_xlabel('False Positive Rate (FPR)')
ax.set_ylabel('False Negative Rate (FNR)')
ax.set_title('DET Curves - All Models')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 0.5])
ax.set_ylim([0, 0.5])

plt.tight_layout()
plt.savefig(fig_dir / 'det_curves.png', dpi=150)
plt.show()
```

